#### SET UP

In [7]:
import sqlite3

con = sqlite3.connect("chicago_analysis.db")
%load_ext sql
%sql sqlite:///chicago_analysis.db

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


## Crime Analysis

#### 1. Which community areas have the most crimes? (Top 10)

In [8]:
%%sql
WITH area_names AS (
    SELECT DISTINCT community_area_number, community_area_name
    FROM SCHOOLS_DATA
)

SELECT 
    a.community_area_name AS "Community Area",
    PRINTF('%,d', COUNT(c.id)) AS "Number of Crimes"
FROM area_names a
JOIN CRIME_DATA c
    ON a.community_area_number = CAST(c.community_area AS INTEGER)
WHERE c.community_area IS NOT NULL
GROUP BY a.community_area_number
ORDER BY COUNT(c.id) DESC
LIMIT 10

 * sqlite:///chicago_analysis.db
Done.


Community Area,Number of Crimes
AUSTIN,"44,123"
SOUTH SHORE,"23,724"
NEAR NORTH SIDE,"22,550"
HUMBOLDT PARK,"22,388"
WEST ENGLEWOOD,"20,830"
NORTH LAWNDALE,"20,417"
WEST TOWN,"19,836"
AUBURN GRESHAM,"19,709"
NEAR WEST SIDE,"19,103"
ROSELAND,"18,965"


##### Data Quality Investigation Queries*
###### *Refer to README file

In [ ]:
%sql SELECT community_area FROM CRIME_DATA ORDER BY community_area NULLS FIRST LIMIT 100 OFFSET 100

In [ ]:
%sql SELECT COUNT(*) - COUNT(community_area) AS null_count FROM CRIME_DATA

In [ ]:
%sql SELECT community_area, COUNT(*) AS crimes_count FROM CRIME_DATA GROUP BY community_area

In [ ]:

%%sql
SELECT SUM(crime_count) AS grouped_total
FROM (
    SELECT 
        community_area,
        COUNT(*) AS crime_count
    FROM CRIME_DATA
    WHERE community_area IS NOT NULL
    GROUP BY community_area
)

In [ ]:
%%sql

SELECT DISTINCT c.community_area
FROM CRIME_DATA c
LEFT JOIN SCHOOLS_DATA s
    ON CAST(c.community_area AS INTEGER) = s.community_area_number
WHERE s.community_area_number IS NULL
AND c.community_area IS NOT NULL
ORDER BY c.community_area

In [ ]:
%%sql

SELECT COUNT(*) AS unmatched_crimes
FROM CRIME_DATA c
LEFT JOIN SCHOOLS_DATA s
    ON CAST(c.community_area AS INTEGER) = s.community_area_number
WHERE s.community_area_number IS NULL
AND c.community_area IS NOT NULL

#### 2. What are the top 5 most common crime types?

In [ ]:
%%sql 
SELECT primary_type AS crime_type, PRINTF('%,d', COUNT(*)) AS crimes_count 
FROM CRIME_DATA 
GROUP BY primary_type
ORDER BY COUNT(*) DESC
LIMIT 5

#### 3. What % of crimes result in arrest?

In [ ]:
%%sql 
WITH crimes AS (
    SELECT COUNT(*) AS total_crimes
    FROM CRIME_DATA)
SELECT ROUND(((COUNT(*) * 100.0) / c.total_crimes), 2) AS percentage_crimes_arrests 
FROM crimes c, CRIME_DATA
WHERE arrest = 1

#### 4. Which locations (location_description) are have the most crimes? (Top 5)

In [ ]:
%%sql 
SELECT 
    location_description AS dangerous_locations,
    COUNT(*) AS crime_count
FROM CRIME_DATA 
WHERE location_description IS NOT NULL
GROUP BY location_description
ORDER BY crime_count DESC
LIMIT 5

###### *Streets and sidewalks account for the largest share of crimes among known location types, suggesting public spaces are a significant, though not necessarily majority crime hotspot. Residences and apartments together also represent a substantial portion, indicating that private spaces are nearly as crime-prone as public ones.

#### 5. Are crimes more domestic in high hardship areas?

In [ ]:
%%sql 

WITH area_names AS (
    SELECT DISTINCT community_area_number, community_area_name
    FROM SCHOOLS_DATA
)

SELECT 
    a.community_area_name AS "Community Area",
    PRINTF('%,d', COUNT(c.id)) AS "Number of Crimes",
    cd.hardship_index AS "Hardship Index"
FROM area_names a
JOIN CRIME_DATA c
    ON a.community_area_number = CAST(c.community_area AS INTEGER)
JOIN CENSUS_DATA cd 
    ON c.community_area = cd.ca
WHERE c.community_area IS NOT NULL
AND UPPER(c.description) LIKE '%DOMESTIC%'
GROUP BY a.community_area_number
ORDER BY COUNT(c.id) DESC
LIMIT 10

###### *I Don't see a direct correlation, here we are showing the top 10 areas with the most crimes and the hardship indexes go up and down, while 60% of them are on the higher side the rest are not, there isn't a set pattern for this hypothesis.

## School Analysis

#### 1. What is the average safety score across all schools?

In [ ]:
%%sql 
SELECT ROUND(AVG(safety_score), 2) AS "Average Safety Score"
FROM SCHOOLS_DATA
WHERE safety_score IS NOT NULL

#### 2. Which school types (Elementary/Middle/High) perform best?

In [ ]:
%sql SELECT MAX(isat_exceeding_math_), MAX(isat_exceeding_reading_) FROM SCHOOLS_DATA


In [ ]:
%%sql
WITH school_stats AS (
    SELECT 
        elementary_or_high_school AS school_type,
        AVG(isat_exceeding_math_) AS math_avg,
        AVG(isat_exceeding_reading_) AS reading_avg
    FROM SCHOOLS_DATA
    GROUP BY elementary_or_high_school
)

SELECT 
    school_type,
    ROUND(math_avg, 2) AS math_isat_average,
    ROUND(reading_avg, 2) AS reading_isat_average,
    ROUND((math_avg + reading_avg) / 2, 2) AS best
FROM school_stats
ORDER BY best DESC

####  3. Top 10 schools by college enrollment

In [ ]:
%%sql
SELECT 
    name_of_school,
    CAST(NULLIF(college_enrollment_rate, 'NDA') AS REAL) AS college_enrollment_rt
FROM SCHOOLS_DATA
ORDER BY college_enrollment_rt DESC
LIMIT 10

#### 4. Schools with safety score below average

In [ ]:
%%sql
WITH safety_stats AS(
    SELECT 
        AVG(safety_score) AS safety_score_avg
    FROM SCHOOLS_DATA
)
SELECT 
    name_of_school, 
    safety_score
FROM SCHOOLS_DATA, safety_stats ss
WHERE safety_score < ss.safety_score_avg
ORDER BY safety_score

#### 5. Is there a pattern between attendance and safety score?

In [ ]:
%%sql 
SELECT 
    average_student_attendance AS min_attendance, 
    safety_score,
    CASE 
        WHEN safety_score >= 70 THEN 'Safe'
        WHEN safety_score >= 50 THEN 'Moderate'
        ELSE 'Unsafe'
    END AS safety_category
FROM SCHOOLS_DATA
WHERE average_student_attendance = (SELECT MIN(average_student_attendance) FROM SCHOOLS_DATA)

In [ ]:
%%sql 
SELECT 
    average_student_attendance AS max_attendance, 
    safety_score,
    CASE 
            WHEN safety_score >= 70 THEN 'Safe'
            WHEN safety_score >= 50 THEN 'Moderate'
            ELSE 'Unsafe'
        END AS safety_category
FROM SCHOOLS_DATA
WHERE average_student_attendance = (SELECT MAX(average_student_attendance) FROM SCHOOLS_DATA)

In [ ]:
%%sql 
SELECT 
    name_of_school,
    average_student_attendance,
    safety_score,
    CASE 
        WHEN safety_score >= 70 THEN 'Safe'
        WHEN safety_score >= 50 THEN 'Moderate'
        ELSE 'Unsafe'
    END AS safetiness
FROM SCHOOLS_DATA
WHERE safety_score IS NOT NULL
ORDER BY safety_score DESC

###### Prior to vizualization data doesnt seem to show any pattern, the lowest attendance school shows a slightly below average safety score, while the highest attendance school shows a even more lower value below the average safety score. *refer to 03_visualizations.ipynb

## Cross Table Analysis

#### 1. Do high hardship areas have more crimes?

In [ ]:
%%sql

WITH crimes_stats AS(
    SELECT
        COUNT(*) AS crime_count,
        community_area AS community
    FROM CRIME_DATA
    WHERE community_area IS NOT NULL
    GROUP BY community_area
)
SELECT 
    cd.community_area_name, 
    cd.hardship_index,
    cs.crime_count,
    CASE
        WHEN cs.crime_count >= AVG(cs.crime_count) OVER () THEN 'Above average'
        ELSE 'Below average'
    END AS crime_qualification
FROM CENSUS_DATA cd
JOIN crimes_stats cs
ON cd.ca = cs.community
WHERE cd.ca IS NOT NULL
AND cd.hardship_index >= 70
ORDER BY cd.hardship_index DESC

###### *High hardship areas do not automatically have more crimes in absolute terms. Extreme hardship communities tend to have very low populations, which suppresses raw crime counts. However, mid-to-high hardship communities with larger populations — like Austin and West Englewood — show significantly higher crime volumes, suggesting that the relationship between hardship and crime is mediated by population size. To properly answer this question, crime rate per capita would be a necessary metric than raw crime counts.

#### 2. Do high hardship areas have lower school safety scores?


In [ ]:
%%sql
SELECT 
    cd.hardship_index,
    ROUND(AVG(sd.safety_score), 2) AS avg_community_safety_score,
    CASE 
        WHEN AVG(sd.safety_score) >= 70 THEN 'Safe'
        WHEN AVG(sd.safety_score) >= 50 THEN 'Moderate'
        ELSE 'Unsafe'
    END AS safetiness
FROM CENSUS_DATA cd 
JOIN SCHOOLS_DATA sd
ON CAST(cd.ca AS INT) = sd.community_area_number
WHERE cd.hardship_index IS NOT NULL
AND sd.safety_score IS NOT NULL
AND cd.hardship_index >= 70
GROUP BY sd.community_area_number
ORDER BY cd.hardship_index DESC


###### *High hardship areas overwhelmingly have lower school safety scores. The data shows a near-uniform "Unsafe" classification across all communities with hardship indexes between 70–98, with average safety scores rarely exceeding 50. 

#### 3. Which community areas appear in BOTH top 10 crime AND bottom 10 school safety?

In [ ]:
%%sql
WITH crimes_stats AS (
    SELECT 
        COUNT(*) AS crimes_count,
        community_area AS community
    FROM CRIME_DATA
    WHERE community_area IS NOT NULL
    GROUP BY community_area
),
top10_crimes AS (
    SELECT 
        community
    FROM crimes_stats
    ORDER BY crimes_count DESC
    LIMIT 10
),
botton10_schools AS (
    SELECT 
        AVG(safety_score) AS community_avg_safety,
        community_area_name,
        community_area_number
    FROM SCHOOLS_DATA
    WHERE safety_score IS NOT NULL
    AND community_area_number IS NOT NULL
    GROUP BY community_area_number
    ORDER BY community_avg_safety ASC
    LIMIT 10
)

SELECT 
    DISTINCT b.community_area_name
FROM botton10_schools b
JOIN top10_crimes t
ON b.community_area_number = t.community




#### 4. Hardship index vs average college enrollment by community area.

In [ ]:
%%sql
SELECT 
    s.community_area_name,
    ROUND(AVG(s.college_enrollment_rate), 2) AS avg_college_enrollment_rate,
    CASE
        WHEN AVG(s.college_enrollment_rate) >= 70 THEN 'HIGH'
        WHEN AVG(s.college_enrollment_rate) >= 50 THEN 'MODERATE'
        ELSE 'LOW'
    END AS avg_college_enrollment_rate_label,
    c.hardship_index,
    CASE
        WHEN AVG(c.hardship_index) >= 70 THEN 'HIGH'
        WHEN AVG(c.hardship_index) >= 50 THEN 'MODERATE'
        ELSE 'LOW'
    END AS hardship_index_label
FROM SCHOOLS_DATA s
JOIN CENSUS_DATA c
ON CAST(c.ca AS INT) = s.community_area_number
WHERE s.college_enrollment_rate IS NOT 'NDA'
AND c.ca IS NOT NULL 
AND s.community_area_number IS NOT NULL 
AND c.hardship_index IS NOT NULL
AND s.community_area_name IS NOT NULL
GROUP BY s.community_area_number
ORDER BY avg_college_enrollment_rate DESC

###### *There is a general negative relationship between hardship index and college enrollment — low hardship communities tend to cluster at the top of enrollment rankings, while high hardship communities dominate the bottom. However, the relationship is not deterministic. Several high-hardship communities achieve moderate enrollment rates, and some low-hardship communities underperform. This suggests that while socioeconomic hardship is a significant barrier to college enrollment, it is not the sole factor

#### 5. Which community area is the most "at risk" across all 3 metrics?

In [ ]:
%%sql
WITH crimes_stats AS (
    SELECT 
        COUNT(*) crimes_count,
        community_area
    FROM CRIME_DATA
    WHERE community_area IS NOT NULL
    GROUP BY community_area
),
school_stats AS (
    SELECT 
        ROUND(AVG(safety_score), 2) AS avg_school_safety_score,
        community_area_number,
        community_area_name
    FROM SCHOOLS_DATA
    WHERE safety_score IS NOT NULL
    AND community_area_number IS NOT NULL 
    AND community_area_name IS NOT NULL
    GROUP BY community_area_number 
)
SELECT 
    ss.community_area_name,
    ss.avg_school_safety_score,
    cs.crimes_count,
    cd.hardship_index
FROM school_stats ss
JOIN crimes_stats cs 
ON ss.community_area_number = CAST(cs.community_area AS INT)
JOIN CENSUS_DATA cd
ON cs.community_area = cd.ca
ORDER BY crimes_count DESC, hardship_index DESC, avg_school_safety_score ASC
LIMIT 1